In [ ]:
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.interpolate import UnivariateSpline

In [ ]:
def process_eyelink_data(file_path):

    """
    Reads and preprocesses EyeLink data from an Excel (or CSV) file.
    
    Parameters:
        file_path (str): The path to the file to be read.
    Returns:
        pd.DataFrame: The processed dataframe.
    """
    
    # Load the data
    # If your raw Eyelink data is in Text/TSV format, use pd.read_csv(file_path, header=None, sep='\t') instead
    df = pd.read_excel(file_path, header=None)
    
    # 1. Delete rows above the first numeric row in Column A
    is_numeric = pd.to_numeric(df[0], errors='coerce').notna()
    if not is_numeric.any():
        raise ValueError("No numeric data found in Column A.")
        
    first_num_idx = is_numeric.idxmax()
    df = df.iloc[first_num_idx:].reset_index(drop=True)
    
    # Recompute the is_numeric mask since the index was reset
    is_numeric = pd.to_numeric(df[0], errors='coerce').notna()
    
    # 2. Keep only Columns A, D, and G (Indices 0, 3, 6)
    cols_to_keep = [0, 3, 6]
    cols_to_keep = [c for c in cols_to_keep if c < df.shape[1]]
    df = df.iloc[:, cols_to_keep].copy()
    df.columns = [0, 1, 2] # Temporarily rename columns to 0, 1, 2 for easy handling
    
    # 3. Get the original data's step width (stepMs)
    numeric_vals = pd.to_numeric(df[0][is_numeric], errors='coerce').values
    step_ms = 2.0
    if len(numeric_vals) >= 2:
        step_ms = float(numeric_vals[1] - numeric_vals[0])
        if step_ms <= 0:
            step_ms = 2.0
            
    # Initialize category columns with 0 across all rows
    df['blue_active'] = 0
    df['green_active'] = 0
    
    # 4. Search for the first 'INPUT'
    is_input = df[0].astype(str).str.contains('INPUT', case=False, na=False)
    if not is_input.any():
        raise ValueError("No 'INPUT' found in Column A.")
        
    first_input_idx = is_input.idxmax()
    
    # 5. Set the starting position (startRow) to 250 numeric data points BEFORE the first 'INPUT'
    numeric_indices_before = is_numeric.iloc[:first_input_idx]
    numeric_indices_before = numeric_indices_before[numeric_indices_before].index
    
    start_idx = numeric_indices_before[-250] if len(numeric_indices_before) >= 250 else (numeric_indices_before[0] if len(numeric_indices_before) > 0 else 0)
    
    # 6. Target 5000 numeric data points starting from the starting position (endRow)
    numeric_indices_after = is_numeric.iloc[start_idx:]
    numeric_indices_after = numeric_indices_after[numeric_indices_after].index
    
    end_idx = numeric_indices_after[4999] if len(numeric_indices_after) >= 5000 else (numeric_indices_after[-1] if len(numeric_indices_after) > 0 else df.index[-1])
    
    # 7. Assign Categories (Toggle between A and B every time 'INPUT' appears)
    # We use a fast, vectorized approach using cumsum() to count INPUT occurrences.
    # Even counts flag Category A, odd counts flag Category B.
    input_counts = is_input.loc[start_idx:end_idx].cumsum()
    
    df.loc[start_idx:end_idx, 'blue_active'] = (input_counts % 2 == 0).astype(int)
    df.loc[start_idx:end_idx, 'green_active'] = (input_counts % 2 == 1).astype(int)
    
    # 8. Delete non-numeric string rows in Column A (Keep only numeric rows)
    df = df[is_numeric].copy()
    
    # 9. Completely reassign the elapsed time
    df['Elapsed_Time'] = np.arange(1, len(df) + 1) * step_ms
    
    # 10. Set final column structure and header names
    final_df = pd.DataFrame({
        'Elapsed Time (ms)': df['Elapsed_Time'].astype(int), # Cast to int to mimic VBA's NumberFormat = "0"
        'Category (A)': df['blue_active'],
        'Category (B)': df['green_active'],
        'Left Eye Pupil Area': df[1],
        'Right Eye Pupil Area': df[2]
    })
    
    print("Processing complete.")
    return final_df


In [ ]:
df = process_eyelink_data('../../../data/260521_1_during1.xlsm')
df